In [ ]:
import os
from bs4 import BeautifulSoup
import pandas as pd

from bs4 import BeautifulSoup
import pandas as pd

# abrir directamente el archivo local
with open("index.html", "r", encoding="utf-8") as f:
    contenido_html = f.read()


soup = BeautifulSoup(contenido_html, "html.parser")

# 3. Pasar el contenido a BeautifulSoup para analizarlo
soup = BeautifulSoup(contenido_html, "html.parser")

# Seleccionar las 20 ofertas de trabajo simuladas
# 
ofertas_html = soup.select("article.oferta-trabajo")

lista_extraccion = []

# 5. Extraer las variables 
for oferta in ofertas_html:
    titulo = oferta.find("h2", class_="titulo-puesto").get_text().strip()
    empresa = oferta.find("div", class_="empresa").get_text().strip()
    descripcion = oferta.find("p", class_="descripcion").get_text().strip()
    salario_texto = oferta.find("p", class_="salario").get_text().strip()
    
    lista_extraccion.append({
        "Título": titulo,
        "Empresa": empresa,
        "Descripción": descripcion,
        "Salario_Original": salario_texto
    })

# 6. CREAR EL DATAFRAME DE PANDAS
df_empleos = pd.DataFrame(lista_extraccion)


# ANÁLISIS DE FRAUDE 


# Limpiar el salario para volverlo un número estadístico
# Quitamos "Salario: S/. ", los puntos y las comas, y quitamos " al mes"
df_empleos["Salario_Numerico"] = (
    df_empleos["Salario_Original"]
    .str.replace("Salario: S/. ", "")
    .str.replace(" al mes", "")
    .str.replace(",", "")
    .astype(float)
)

# Aplicar un criterio para detectar fraudes
# Según los patrones, las estafas ofrecen salarios absurdamente altos por trabajos simples.
# Definimos como "Sospechosa de Fraude" cualquier oferta que pague más de S/. 5,000 al mes.
Sueldo_Maximo_Normal = 5000.0
df_empleos["¿Es_Fraude?"] = df_empleos["Salario_Numerico"] > Sueldo_Maximo_Normal

# Paso C: Separar las ofertas legítimas de las estafas para el reporte
ofertas_legitimas = df_empleos[df_empleos["¿Es_Fraude?"] == False]
ofertas_fraudulentas = df_empleos[df_empleos["¿Es_Fraude?"] == True]

# 7. Mostrar los resultados

print("      REPORTE ESTADÍSTICO DE DETECCIÓN DE FRAUDES      ")

print(f"Total de ofertas analizadas: {len(df_empleos)}")
print(f"Ofertas Legítimas detectadas: {len(ofertas_legitimas)}")
print(f"Ofertas FRAUDULENTAS detectadas: {len(ofertas_fraudulentas)}")
print("=======================================================\n")

print(" LISTA DE EMPRESAS Y PUESTOS BAJO SOSPECHA:")
for index, fila in ofertas_fraudulentas.iterrows():
    print(f"- Puesto: '{fila['Título']}' | Ofrece: {fila['Salario_Original']} | Empresa: {fila['Empresa']}")

# Guardar la base de datos consolidada en un CSV para subirlo a GitHub
df_empleos.to_csv("reporte_final_empleos.csv", index=False, encoding="utf-8")